In [ ]:
import torch
from models import TrainableModelBase
from dataclasses import dataclass
from utils import MatrixNormalization


class SimpleLinear(TrainableModelBase):
    @dataclass
    class ModelConfig(TrainableModelBase.ModelConfig):
        power: float = 0.0
        gamma: float = 0.0
        normalization_method: str = 'zscore'

    def __init__(
        self,
        user_matrix: torch.Tensor,
        item_matrix: torch.Tensor,
        rate_matrix: torch.Tensor,
        model_config: ModelConfig,
    ):
        super().__init__(model_config=model_config)
        self.rate_matrix = rate_matrix

        

    @torch.no_grad()
    def predict(self) -> torch.Tensor:
        normalize_fn = MatrixNormalization.get_method(self.model_config.normalization_method)

        # 语义预测矩阵
        se_pred_matrix = (
            self.svd_U[:self.num_users, :len(self.svd_S)] * torch.pow(self.svd_S, self.model_config.power)
            @ (self.svd_U[self.num_users:, :len(self.svd_S)] * torch.pow(self.svd_S, self.model_config.power)).T
        )

        # 归一化并融合
        norm_cf = normalize_fn(self.cf_pred_matrix)
        norm_se = normalize_fn(se_pred_matrix)

        gamma = self.model_config.gamma
        return (1 - gamma) * norm_cf + gamma * norm_se
